# 22.6 Kubernetes 入门:模型服务的编排 / Kubernetes Basics for ML

**中文**:22.5 的 Docker 把模型服务打包成了镜像,能在**一台机器上跑一个容器**。但生产环境需要更多:**几十个副本分担流量、某个容器崩了自动重启、流量高峰自动扩容、发新版本时不中断服务(滚动更新)、负载均衡**……手动管理成百上千个容器是不可能的。**Kubernetes(K8s)** 就是干这个的——**容器编排的事实标准**:你只需**声明"我想要什么"**(比如"我要 3 个这个模型服务的副本,永远保持"),K8s 就持续地让现实**收敛到你的期望**。它最核心的思想——**声明式 + 调和循环(reconciliation loop)**——优雅得惊人。本节从零实现这个调和循环,让你亲眼看到 K8s 如何**自愈**(pod 崩了自动重建)和**扩缩容**,再给出真实的 K8s YAML。
**English**: 22.5's Docker packaged the model service into an image that **runs one container on one machine**. But production needs more: **dozens of replicas sharing traffic, auto-restart when a container crashes, auto-scale at traffic peaks, zero-downtime deploys (rolling updates), load balancing**… manually managing hundreds or thousands of containers is impossible. **Kubernetes (K8s)** does exactly this — the **de-facto standard for container orchestration**: you just **declare "what you want"** (e.g. "I want 3 replicas of this model service, always"), and K8s continuously makes reality **converge to your desire**. Its core idea — **declarative + reconciliation loop** — is astonishingly elegant. This section builds that reconciliation loop from scratch so you see how K8s **self-heals** (recreates a crashed pod) and **scales**, then gives real K8s YAML.

---

**中文**:**K8s 的核心对象(ML 部署要会这几个)**:
**English**: **K8s's core objects (know these for ML deployment)**:
- **中文**:**Pod**:最小调度单位,包一个(或几个紧密相关的)容器。你的模型服务容器就跑在 Pod 里。Pod 是**易逝的**(会被杀、被重建)。
  **Pod**: the smallest scheduling unit, wrapping one (or a few tightly-related) containers. Your model service container runs in a Pod. Pods are **ephemeral** (killed and recreated).
- **中文**:**Deployment**:声明"我要 N 个某镜像的 Pod 副本",并管理它们的生命周期(自愈、扩缩、**滚动更新**)。你几乎总是通过 Deployment 而非直接创建 Pod。
  **Deployment**: declares "I want N Pod replicas of some image" and manages their lifecycle (self-healing, scaling, **rolling updates**). You almost always go through a Deployment, not raw Pods.
- **中文**:**Service**:给一组易逝的 Pod 一个**稳定的网络入口 + 负载均衡**(Pod 换来换去,Service 的地址不变)。客户端访问 Service,流量被分发到后面的 Pod。
  **Service**: gives a set of ephemeral Pods a **stable network endpoint + load balancing** (Pods come and go, the Service address stays). Clients hit the Service; traffic is distributed to the Pods behind it.
- **中文**:**HPA(Horizontal Pod Autoscaler)**:根据 CPU/QPS 等指标**自动增减副本数**(流量涨就扩、跌就缩)。
  **HPA (Horizontal Pod Autoscaler)**: **auto-adjusts the replica count** by metrics like CPU/QPS (scale up when traffic rises, down when it falls).

**中文**:**K8s 最核心、最优雅的思想:声明式 + 调和循环(reconciliation loop)**。你不告诉 K8s "怎么做"(命令式:先起一个 pod、再起一个……),而是声明**期望状态**(desired state):"我要 3 个副本"。K8s 的**控制器(controller)** 跑一个永不停歇的循环:*不断对比"期望状态"和"实际状态",一旦有差异,就采取行动让实际收敛到期望*。Pod 崩了 → 实际(2)< 期望(3)→ 控制器新建一个 → 回到 3(**自愈**);你把期望改成 5 → 实际(3)< 期望(5)→ 新建 2 个(**扩容**)。这个"声明期望 + 自动收敛"的模型,就是 K8s 强大与可靠的根源。
**English**: **K8s's most core and elegant idea: declarative + reconciliation loop.** You don't tell K8s "how to do it" (imperative: start one pod, then another…), but declare the **desired state**: "I want 3 replicas." K8s's **controller** runs an endless loop: *continuously compare "desired state" vs "actual state," and whenever they differ, act to converge actual toward desired*. A Pod crashes → actual (2) < desired (3) → the controller creates one → back to 3 (**self-healing**); you change desired to 5 → actual (3) < desired (5) → create 2 (**scaling**). This "declare desire + auto-converge" model is the source of K8s's power and reliability.

> 💡 **面试速查 / Interview cheat-sheet（★★ 云原生/部署必考）**
> **中文**:**K8s**=容器编排事实标准, 管理集群里成百上千容器(扩缩/自愈/滚动更新/负载均衡)。**核心对象**:**Pod**(最小单位, 包容器, 易逝)→**Deployment**(管 N 个 Pod 副本, 自愈+滚动更新)→**Service**(稳定入口+负载均衡, Pod 易逝但 Service 地址不变)→**Ingress**(对外 HTTP 路由)→**HPA**(按指标自动扩缩)→**ConfigMap/Secret**(配置/密钥)→**PVC**(持久卷)。**灵魂=声明式+调和循环**:你声明期望状态, 控制器持续对比实际并驱动收敛→自愈/扩缩/滚动更新都是它。**探针**:liveness(死了重启)/readiness(没准备好不给流量, 接 22.3 的 /health)。**ML 特有**:**KServe/Seldon**(K8s 上的模型服务, 自带自动扩缩到0、金丝雀、批处理、GPU 调度)、**Kubeflow**(K8s 上的 ML 平台/流水线)、GPU 调度(nvidia device plugin)。**vs Docker**:Docker 打单机容器, K8s 编排集群多容器。面试金句:*"K8s 是声明式容器编排:你声明期望状态(如 Deployment 要 3 副本), 控制器的调和循环持续对比实际状态并收敛, 从而自愈(pod 崩了重建)、扩缩(HPA)、滚动更新(不停机发版); Pod 是易逝最小单位、Service 给稳定入口和负载均衡; ML 上用 KServe/Kubeflow 做模型服务和流水线, 配 liveness/readiness 探针。"*
> **English**: **K8s** = the de-facto container orchestration standard, managing hundreds/thousands of containers in a cluster (scaling/self-healing/rolling updates/load balancing). **Core objects**: **Pod** (smallest unit, wraps containers, ephemeral) → **Deployment** (manages N Pod replicas, self-healing + rolling updates) → **Service** (stable endpoint + load balancing, Pods are ephemeral but the Service address is stable) → **Ingress** (external HTTP routing) → **HPA** (auto-scale by metrics) → **ConfigMap/Secret** (config/secrets) → **PVC** (persistent volume). **Soul = declarative + reconciliation loop**: you declare desired state, controllers continuously compare actual and drive convergence → self-healing/scaling/rolling updates all come from it. **Probes**: liveness (dead → restart) / readiness (not ready → no traffic, ties to 22.3's /health). **ML-specific**: **KServe/Seldon** (model serving on K8s, with scale-to-zero, canary, batching, GPU scheduling), **Kubeflow** (ML platform/pipelines on K8s), GPU scheduling (nvidia device plugin). **vs Docker**: Docker builds a single-machine container, K8s orchestrates many across a cluster. Interview line: *"K8s is declarative container orchestration: you declare desired state (e.g. a Deployment wants 3 replicas), and controllers' reconciliation loop continuously compares actual state and converges, giving self-healing (recreate crashed pods), scaling (HPA), and rolling updates (zero-downtime deploys); Pods are ephemeral smallest units, Services give a stable endpoint and load balancing; for ML use KServe/Kubeflow for serving and pipelines with liveness/readiness probes."*


In [ ]:

# ============================================================
# 从零实现 K8s 的调和循环:声明式自愈 + 扩缩容 / K8s reconciliation loop: declarative self-healing + scaling
# 中文:K8s 的灵魂。控制器不断对比"期望副本数"和"实际运行数", 有差异就创建/删除 Pod 让二者收敛。
#      你只声明期望, 剩下的 K8s 自动搞定——这就是声明式编排。
# English: K8s's soul. The controller continuously compares "desired replicas" vs "actual running" and creates/deletes
#      Pods to converge. You only declare the desire; K8s does the rest — declarative orchestration.
# ============================================================
import itertools
_ids=itertools.count(1)
class Cluster:                                              # 模拟集群 / a mini cluster
    def __init__(self): self.pods=set()
    def create_pod(self): pid=f"pod-{next(_ids)}"; self.pods.add(pid); return pid
    def kill_pod(self, pid): self.pods.discard(pid)         # 模拟崩溃/节点故障 / simulate crash/node failure
    def running(self): return sorted(self.pods)

def reconcile(cluster, desired):                           # ★ 控制器核心循环:驱动实际→期望 / the controller's core loop
    actual=cluster.running(); gap=desired-len(actual); acts=[]
    if gap>0:                                               # 太少 → 创建(扩容/自愈)/ too few → create (scale up / self-heal)
        for _ in range(gap): acts.append(("create", cluster.create_pod()))
    elif gap<0:                                             # 太多 → 删除(缩容)/ too many → delete (scale down)
        for pid in actual[:-gap]: cluster.kill_pod(pid); acts.append(("delete", pid))
    return acts

c=Cluster()
print("① Deployment 声明期望 3 个副本 / desired=3:")
print("   reconcile →", reconcile(c, 3), "| 运行中 running:", len(c.running()))
print("\n② 自愈 / self-healing:一个 pod 崩溃")
dead=c.running()[0]; c.kill_pod(dead); print(f"   {dead} 崩溃! 实际只剩 {len(c.running())} 个")
print("   控制器 reconcile →", reconcile(c, 3), "| 运行中:", len(c.running()), "← 自动重建, 恢复到期望的3")
print("\n③ 扩容 / scale up:HPA 把期望改为 5(流量上涨)")
print("   reconcile →", reconcile(c, 5), "| 运行中:", len(c.running()))
print("\n④ 缩容 / scale down:期望改为 2(流量回落)")
print("   reconcile →", reconcile(c, 2), "| 运行中:", len(c.running()))
print("\n核心:你从不手动管 Pod, 只声明期望状态; 控制器的调和循环持续对比并自动收敛(自愈+扩缩)")


**中文**:上面是原理。下面是把 22.3/22.5 的模型服务部署到 K8s 的**真实 YAML**——注意它是**声明式**的(描述"要什么"而非"怎么做"):
**English**: The above is the principle. Below is the **real YAML** to deploy 22.3/22.5's model service to K8s — note it's **declarative** (describing "what" not "how"):

```yaml
# deployment.yaml —— 部署 / deploy:  kubectl apply -f deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata: { name: iris-api }
spec:
  replicas: 3                          # ★ 声明期望:永远保持 3 个副本 / desired state: always keep 3 replicas
  selector: { matchLabels: { app: iris-api } }
  template:
    metadata: { labels: { app: iris-api } }
    spec:
      containers:
      - name: iris-api
        image: myregistry/iris-api:v1.2   # 22.5 构建并推送的镜像 / the image built & pushed in 22.5
        ports: [ { containerPort: 8000 } ]
        resources:                        # 资源请求/上限, 供调度和 HPA / resource requests/limits
          requests: { cpu: "250m", memory: "512Mi" }
          limits:   { cpu: "1",    memory: "1Gi" }
        readinessProbe:                   # 就绪探针(接 22.3 的 /health): 没准备好就不给流量 / no traffic until ready
          httpGet: { path: /health, port: 8000 }
          initialDelaySeconds: 5
        livenessProbe:                    # 存活探针: 卡死了就重启这个 pod / restart if dead
          httpGet: { path: /health, port: 8000 }
          periodSeconds: 10
---
apiVersion: v1
kind: Service                            # 给 3 个易逝的 Pod 一个稳定入口 + 负载均衡 / stable endpoint + LB
metadata: { name: iris-api-svc }
spec:
  selector: { app: iris-api }
  ports: [ { port: 80, targetPort: 8000 } ]
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler            # 按 CPU 自动扩缩 3~10 副本 / autoscale 3–10 by CPU
metadata: { name: iris-api-hpa }
spec:
  scaleTargetRef: { apiVersion: apps/v1, kind: Deployment, name: iris-api }
  minReplicas: 3
  maxReplicas: 10
  metrics: [ { type: Resource, resource: { name: cpu, target: { type: Utilization, averageUtilization: 70 } } } ]
```
**中文**:你 `kubectl apply` 这份 YAML,K8s 就接管一切:保持 3 副本、崩了重建、CPU 超 70% 就扩容、发新版本滚动更新——全是"声明期望、自动收敛"。
**English**: You `kubectl apply` this YAML, and K8s takes over: keep 3 replicas, recreate on crash, scale up above 70% CPU, roll out new versions — all "declare desire, auto-converge."


In [ ]:

# ============================================================
# 可视化:调和循环(自愈+扩缩)/ reconciliation loop (self-healing + scaling)
# ============================================================
import matplotlib.pyplot as plt
# 模拟一段时间内 期望 vs 实际 副本数 / simulate desired vs actual over time
timeline=[]; cl=Cluster(); desired=3
events={2:"pod崩溃", 4:"HPA扩容→5", 7:"HPA缩容→2", 9:"pod崩溃"}
for t in range(11):
    if t==4: desired=5
    if t==7: desired=2
    if t in (2,9) and cl.running(): cl.kill_pod(cl.running()[0])   # inject a crash before reconcile
    reconcile(cl, desired)                                          # controller acts
    timeline.append((t, desired, len(cl.running())))
fig,ax=plt.subplots(1,2,figsize=(14,5))
ts=[x[0] for x in timeline]
ax[0].step(ts,[x[1] for x in timeline],where="post",lw=2,color="#4C72B0",label="期望副本 desired")
ax[0].step(ts,[x[2] for x in timeline],where="post",lw=2,ls="--",color="#55A868",label="实际副本 actual(收敛后)")
for t,lab in events.items(): ax[0].axvline(t,color="#C44E52",alpha=0.3); ax[0].text(t,5.3,lab,rotation=90,fontsize=7,color="#C44E52")
ax[0].set_xlabel("时间(调和周期)"); ax[0].set_ylabel("副本数"); ax[0].set_title("调和循环:实际持续收敛到期望"); ax[0].legend(fontsize=9)
# K8s 对象层级 / object hierarchy
ax[1].axis("off"); ax[1].set_title("K8s 对象层级",fontsize=12,weight="bold")
boxes=[("HPA (按指标自动改期望副本数)",0.78,"#9467BD"),("Deployment (管 N 个副本, 自愈+滚动更新)",0.6,"#4C72B0"),
       ("ReplicaSet → Pods (易逝的容器实例)",0.42,"#DD8452"),("Service (稳定入口 + 负载均衡)",0.2,"#55A868")]
for lab,y,c in boxes:
    ax[1].add_patch(plt.Rectangle((0.1,y),0.8,0.13,fc=c,alpha=0.3,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.5,y+0.065,lab,ha="center",va="center",fontsize=9,transform=ax[1].transAxes)
ax[1].annotate("",xy=(0.5,0.6),xytext=(0.5,0.78),arrowprops=dict(arrowstyle="->"),transform=ax[1].transAxes)
ax[1].annotate("",xy=(0.5,0.42),xytext=(0.5,0.6),arrowprops=dict(arrowstyle="->"),transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/mlops06_viz.png",dpi=80); plt.show()
print("左:无论 pod 崩溃还是期望变化, 实际副本数总被调和循环拉回期望; 右:HPA→Deployment→Pods, Service 提供稳定入口")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **K8s 的全部威力,来自"声明式 + 调和循环"这一个思想**:我们几十行就复现了它的心脏——你**只声明"我要 3 个副本"**,一个永不停歇的控制器不断对比期望与实际、自动补齐或删减。这一个机制,同时给了你:**自愈**(pod 崩了自动重建,不用半夜起来重启服务)、**扩缩容**(HPA 改期望值,流量涨自动加副本)、**滚动更新**(发新版本时,逐个替换 pod、旧的还在服务、新的准备好才切,零停机)。命令式("先干这个再干那个")的运维脚本脆弱易错,而声明式("我要什么样")让系统自己维持你要的状态——这是现代基础设施(不只是 K8s,还有 Terraform、GitOps)的统一哲学:**描述期望,让系统收敛**。
2. **对 ML 部署,K8s 解决的是"规模化的可靠性"**:单个 FastAPI 容器(22.3/22.5)能跑,但真实生产要面对:流量波动(半夜没人、白天高峰)、机器故障、发新模型版本不能中断、多个模型/多副本的负载均衡。K8s 把这些**变成配置而非人肉运维**:`replicas`、`HPA`、`readinessProbe`(接 22.3 的 `/health`——没准备好就不给流量,正是这里用上的)、滚动更新策略。ML 领域还有专门的封装:**KServe / Seldon**(在 K8s 上一键部署模型,自带自动扩缩到 0、金丝雀发布、请求批处理、GPU 调度)、**Kubeflow**(K8s 上的完整 ML 平台,跑训练流水线)。
3. **诚实的边界:K8s 极其强大,但也极其复杂——别为小事上 K8s**。①**复杂度税**:K8s 有陡峭的学习曲线和巨大的运维负担(网络、存储、RBAC 权限、Ingress、证书、监控……),一个"Hello World"服务背后是几百行 YAML 和一堆概念。**如果你只是要跑一两个小服务,K8s 是杀鸡用牛刀**——用云厂商的托管容器服务(AWS Fargate、Google Cloud Run、Azure Container Apps)或 PaaS 往往更简单。②**它的价值在规模**:当你有几十上百个服务、需要精细的资源调度、多团队共享集群、混合负载时,K8s 的统一编排才真正物有所值。③**"要不要自己运维 K8s"几乎总是否定的**——用云托管的 K8s(EKS/GKE/AKS),别自己搭控制平面(那是专职平台团队的活)。④对数据科学家:**你通常不需要成为 K8s 专家,但要理解它的心智模型**(声明式、Pod/Deployment/Service、探针、HPA),能读懂 YAML、会 `kubectl` 基本命令、知道 KServe/Kubeflow 的存在——这足以让你和平台团队高效协作,并在面试里讲清"模型怎么上生产"。**结论:K8s 用声明式调和循环把'规模化容器运维'自动化,是云原生部署的地基;理解它的心智模型是 MLOps 硬技能,但要清醒它的复杂度——按规模选择(小用托管 PaaS,大才上 K8s),且优先用云托管而非自建。**

**English**:
1. **All of K8s's power comes from one idea: declarative + reconciliation loop**: we reproduced its heart in a few dozen lines — you **only declare "I want 3 replicas,"** and an endless controller continuously compares desired vs actual and auto-fills or trims. This one mechanism simultaneously gives you: **self-healing** (crashed pods auto-recreated, no 3 AM restarts), **scaling** (HPA changes the desire, traffic rises → auto add replicas), **rolling updates** (deploy a new version by replacing pods one by one, old ones still serving, switch only when new ones are ready — zero downtime). Imperative ("do this, then that") ops scripts are fragile and error-prone, while declarative ("what I want") lets the system maintain your desired state itself — the unifying philosophy of modern infrastructure (not just K8s but Terraform, GitOps): **describe the desire, let the system converge**.
2. **For ML deployment, K8s solves "reliability at scale"**: a single FastAPI container (22.3/22.5) runs, but real production faces: traffic fluctuation (empty at night, peak by day), machine failures, non-disruptive new model versions, load balancing across multiple models/replicas. K8s **turns these into configuration rather than manual ops**: `replicas`, `HPA`, `readinessProbe` (ties to 22.3's `/health` — no traffic until ready, used exactly here), rolling-update strategy. ML also has dedicated wrappers: **KServe / Seldon** (one-click model deployment on K8s, with scale-to-zero, canary releases, request batching, GPU scheduling), **Kubeflow** (a full ML platform on K8s running training pipelines).
3. **Honest limits: K8s is extremely powerful but extremely complex — don't use K8s for small things**. ① **Complexity tax**: K8s has a steep learning curve and huge operational burden (networking, storage, RBAC, Ingress, certificates, monitoring…); a "Hello World" service hides hundreds of lines of YAML and a pile of concepts. **If you just need one or two small services, K8s is overkill** — a cloud managed container service (AWS Fargate, Google Cloud Run, Azure Container Apps) or PaaS is often simpler. ② **Its value is at scale**: when you have dozens/hundreds of services, need fine-grained resource scheduling, multi-team shared clusters, mixed workloads, K8s's unified orchestration truly pays off. ③ **"Should I self-operate K8s" is almost always no** — use cloud-managed K8s (EKS/GKE/AKS), don't run your own control plane (that's a dedicated platform team's job). ④ For data scientists: **you usually don't need to be a K8s expert, but understand its mental model** (declarative, Pod/Deployment/Service, probes, HPA), read YAML, know basic `kubectl`, and know KServe/Kubeflow exist — enough to collaborate efficiently with the platform team and explain "how models go to production" in interviews. **Conclusion: K8s automates "container ops at scale" with a declarative reconciliation loop, the foundation of cloud-native deployment; understanding its mental model is an MLOps hard skill, but be clear on its complexity — choose by scale (managed PaaS for small, K8s only for large), and prefer cloud-managed over self-hosted.**

> 💼 **实战视角 / Practical angle**
> **中文**:K8s 落地:①**声明式 YAML**:Deployment(副本+滚动更新)+ Service(入口+负载均衡)+ HPA(自动扩缩)+ ConfigMap/Secret(配置/密钥);②**探针必配**:readinessProbe(接 `/health`, 没准备好不给流量)+ livenessProbe(卡死重启)——这是模型服务稳定的关键;③**资源 requests/limits** 要设(供调度和 HPA, 防止 OOM/抢占);④**ML 用 KServe/Seldon** 部署模型(自动扩缩到0省成本、金丝雀、批处理、GPU);**Kubeflow** 跑训练流水线;⑤**用云托管 K8s**(EKS/GKE/AKS), 别自建控制平面;⑥小规模用 Cloud Run/Fargate 这类 serverless 容器更省心。**别做的**:为一两个服务上自建 K8s(复杂度税)。面试金句:*"K8s 声明式编排:你声明期望(Deployment 3 副本、HPA 扩缩), 控制器调和循环持续对比实际并收敛, 实现自愈/扩缩/滚动更新零停机; Pod 易逝, Service 给稳定入口和负载均衡, readiness/liveness 探针接模型的 /health; ML 用 KServe 部署(自动扩缩到0、金丝雀、GPU)、Kubeflow 跑流水线; 但复杂度高, 小规模用托管 PaaS, 大规模才上云托管 K8s。"*
> **English**: K8s in practice: ① **declarative YAML**: Deployment (replicas + rolling updates) + Service (endpoint + LB) + HPA (autoscale) + ConfigMap/Secret (config/secrets); ② **probes are a must**: readinessProbe (hits `/health`, no traffic until ready) + livenessProbe (restart if stuck) — key to model-service stability; ③ **set resource requests/limits** (for scheduling and HPA, prevent OOM/preemption); ④ **for ML use KServe/Seldon** to deploy models (scale-to-zero saves cost, canary, batching, GPU); **Kubeflow** for training pipelines; ⑤ **use cloud-managed K8s** (EKS/GKE/AKS), don't run your own control plane; ⑥ at small scale, serverless containers (Cloud Run/Fargate) are simpler. **Don't**: self-host K8s for one or two services (complexity tax). Interview line: *"K8s is declarative orchestration: you declare desired state (a Deployment's 3 replicas, HPA scaling), and controllers' reconciliation loop continuously compares actual and converges, giving self-healing/scaling/zero-downtime rolling updates; Pods are ephemeral, Services give a stable endpoint and load balancing, readiness/liveness probes hit the model's /health; for ML use KServe to deploy (scale-to-zero, canary, GPU) and Kubeflow for pipelines; but it's complex, so use managed PaaS at small scale and cloud-managed K8s only at large scale."*

---
### 小结 / Summary
- **中文**:K8s=声明式容器编排; 核心是调和循环(持续对比期望 vs 实际并收敛)→自愈、扩缩(HPA)、滚动更新零停机。
- **English**: K8s = declarative container orchestration; core is the reconciliation loop (continuously compare desired vs actual and converge) → self-healing, scaling (HPA), zero-downtime rolling updates.
- **中文**:Pod(易逝最小单位)→Deployment(管副本)→Service(稳定入口+负载均衡); readiness/liveness 探针接 /health。
- **English**: Pod (ephemeral smallest unit) → Deployment (manages replicas) → Service (stable endpoint + LB); readiness/liveness probes hit /health.
- **中文**:ML 用 KServe/Kubeflow; 复杂度高——小规模用托管 PaaS(Cloud Run/Fargate), 大规模用云托管 K8s(别自建)。
- **English**: For ML use KServe/Kubeflow; high complexity — small scale uses managed PaaS (Cloud Run/Fargate), large scale uses cloud-managed K8s (not self-hosted).
